# Type 1 — End-to-End Pipeline Evaluation

Chạy toàn bộ pipeline Type 1 trên bộ dữ liệu `Logic_Based_Educational_Queries.json` và đánh giá kết quả.

In [1]:
# ── 1. CẤU HÌNH ──────────────────────────────────────────────────────────────
VLLM_BASE_URL  = "https://api.iamphuckhang.dev/v1"  # Cloudflare tunnel hoặc IP:port
MODEL_NAME     = "Qwen/Qwen2.5-7B-Instruct-AWQ"     # tên model trên vLLM
VLLM_API_KEY   = "EMPTY"

MAX_TOKENS     = 1024   # max_model_len=4096, giữ budget nhỏ để tránh overflow
TEMPERATURE    = 0.0
TIMEOUT_SEC    = 60.0
MAX_RETRIES    = 2

# Số lần sample để vote (>=3 tốt hơn nhưng chậm hơn)
TRANSLATION_SAMPLES     = 3
SAMPLING_TEMPERATURE    = 0.7

# Bật CoT fallback khi symbolic trả Unknown
ENABLE_COT_FALLBACK     = True

# None = chạy hết dataset; đặt số nguyên để giới hạn (ví dụ 50 để test nhanh)
LIMIT          = 1

# Đường dẫn output
OUTPUT_PATH    = "../artifacts/predictions/type1/eval_run.json"
REPORT_PATH    = "../artifacts/reports/type1_eval_report.json"
ERRORS_PATH    = "../artifacts/reports/type1_eval_errors.csv"

In [2]:
# ── 2. SETUP ─────────────────────────────────────────────────────────────────
import sys, json, time, csv, importlib
from pathlib import Path
from dataclasses import asdict, dataclass
from typing import Any

ROOT = Path().resolve().parent
SRC  = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"root : {ROOT}")
print(f"src  : {SRC}")

# Reload để lấy code mới nhất từ disk (không cần restart kernel)
import exact.llm_client
import exact.logic.llm_translator
import exact.logic.kb
import exact.logic.pipeline
importlib.reload(exact.llm_client)
importlib.reload(exact.logic.llm_translator)
importlib.reload(exact.logic.kb)
importlib.reload(exact.logic.pipeline)

from pydantic import SecretStr
from exact.config import Settings
from exact.llm_client import build_json_client_from_settings
from exact.datasets.dataset import ExactDataset
from exact.router.task_router import TaskRouter
from exact.logic.pipeline import run_type1_pipeline
from exact.common.schemas import TaskType, to_official_response

print("✓ imports OK")

root : /home/phuckhang/MyWorkspace/Exact2026
src  : /home/phuckhang/MyWorkspace/Exact2026/src
✓ imports OK


In [3]:
# ── 3. KIỂM TRA KẾT NỐI vLLM ────────────────────────────────────────────────
import requests

try:
    r = requests.get(f"{VLLM_BASE_URL}/models", timeout=15)
    if r.status_code == 200:
        models = [m["id"] for m in r.json().get("data", [])]
        print(f"✓ vLLM reachable — models: {models}")
        if MODEL_NAME not in models:
            print(f"⚠ MODEL_NAME='{MODEL_NAME}' không có trong danh sách trên, kiểm tra lại")
    else:
        print(f"✗ HTTP {r.status_code}: {r.text[:200]}")
except Exception as e:
    print(f"✗ Không kết nối được: {e}")
    raise

✓ vLLM reachable — models: ['Qwen/Qwen2.5-7B-Instruct-AWQ']


In [4]:
# ── 4. TẠO SETTINGS VÀ LLM CLIENT ───────────────────────────────────────────
settings = Settings().model_copy(update={
    "llm_provider"              : "openai",
    "llm_base_url"              : VLLM_BASE_URL,
    "llm_model"                 : MODEL_NAME,
    "llm_api_key"               : SecretStr(VLLM_API_KEY),
    "llm_max_tokens"            : MAX_TOKENS,
    "llm_temperature"           : TEMPERATURE,
    "llm_timeout_seconds"       : TIMEOUT_SEC,
    "llm_max_retries"           : MAX_RETRIES,
    "type1_translation_samples" : TRANSLATION_SAMPLES,
    "type1_sampling_temperature": SAMPLING_TEMPERATURE,
    "type1_enable_cot_fallback" : ENABLE_COT_FALLBACK,
    "mock_llm"                  : False,
})

translator_client = build_json_client_from_settings(settings)
print(f"✓ client: {type(translator_client).__name__}")
print(f"  model       : {settings.llm_model}")
print(f"  base_url    : {settings.llm_base_url}")
print(f"  samples     : {settings.type1_translation_samples}")
print(f"  cot_fallback: {settings.type1_enable_cot_fallback}")

✓ client: LLMClient
  model       : Qwen/Qwen2.5-7B-Instruct-AWQ
  base_url    : https://api.iamphuckhang.dev/v1
  samples     : 3
  cot_fallback: True


In [5]:
# ── 5. LOAD DATASET ──────────────────────────────────────────────────────────
DATASET_PATH = ROOT / "src/exact/datasets/exact/Logic_Based_Educational_Queries.json"

dataset  = ExactDataset.from_file(DATASET_PATH, skip_invalid=True).filter_type1()
examples = list(dataset)
if LIMIT is not None:
    examples = examples[:LIMIT]

# Thống kê question type trong tập sẽ chạy
from collections import Counter
from exact.router.task_router import detect_question_type

router = TaskRouter()
qtype_counts: Counter = Counter()
for ex in examples:
    route = router.route(ex.request)
    qtype_counts[route.question_type.value] += 1

print(f"✓ loaded {len(examples)} examples")
print(f"  question types: {dict(qtype_counts)}")
print()
# Xem 1 mẫu
sample = examples[0]
print(f"[sample id]   {sample.request.id}")
print(f"[premises]    {sample.request.premises_nl[:2]}...")
print(f"[question]    {sample.request.question[:120]}")
print(f"[gold_answer] {sample.gold_answer}")

✓ loaded 1 examples
  question types: {'mcq': 1}

[sample id]   logic_0000_00
[premises]    ['If a Python code is well-tested, then the project is optimized.', 'If a Python code does not follow PEP 8 standards, then it is not well-tested.']...
[question]    Which conclusion follows with the fewest premises?
A. If a Python project is not optimized, then it is not well-tested
B
[gold_answer] A


In [6]:
raw = translator_client.complete_json_sync(
    messages=[
        {"role": "system", "content": "Return JSON only."},
        {"role": "user", "content": 'Return {"ok": true}.'},
    ],
    temperature=0.0,
    max_tokens=64,
)
print(raw)

{'ok': True}


In [7]:
# ── 6. CHẠY PIPELINE ─────────────────────────────────────────────────────────
from exact.common.schemas import to_official_response

predictions: list[dict[str, Any]] = []
counters = {"correct": 0, "wrong": 0, "error": 0}
total = len(examples)
t_start = time.time()

for idx, example in enumerate(examples, start=1):
    route = router.route(example.request)
    t0 = time.time()
    try:
        response = run_type1_pipeline(
            example.request,
            translator_client=translator_client,
            settings=settings,
            question_type=route.question_type,
        )
        pred_error = response.error
    except Exception as exc:
        from exact.common.schemas import PredictionResponse, QuestionType
        response = PredictionResponse(
            id=example.request.id,
            task_type=TaskType.TYPE1_LOGIC,
            answer="",
            explanation=f"pipeline error: {exc}",
            confidence=0.0,
            error=str(exc),
        )
        pred_error = str(exc)

    elapsed = time.time() - t0

    # Chấm điểm ngay
    ans      = (response.answer or "").strip().lower()
    gold     = (example.gold_answer or "").strip().lower()
    if pred_error and not ans:
        status = "error"
    elif ans == gold:
        status = "correct"
    else:
        status = "wrong"
    counters[status] += 1

    prediction = response.model_dump(mode="json")
    prediction["gold_answer"]  = example.gold_answer
    prediction["route_reason"] = route.reason
    prediction["official"]     = to_official_response(response)
    prediction["_elapsed_s"]   = round(elapsed, 2)
    prediction["_status"]      = status
    predictions.append(prediction)

    # In progress mỗi 10 item hoặc item cuối
    if idx % 10 == 0 or idx == total:
        scored   = counters["correct"] + counters["wrong"] + counters["error"]
        accuracy = counters["correct"] / scored if scored else 0
        elapsed_total = time.time() - t_start
        eta = (elapsed_total / idx) * (total - idx)
        print(
            f"[{idx:4d}/{total}] "
            f"acc={accuracy:.3f} "
            f"✓{counters['correct']} ✗{counters['wrong']} ⚡{counters['error']} "
            f"| elapsed={elapsed_total:.0f}s ETA={eta:.0f}s"
        )

print(f"\n✓ done — {total} predictions in {time.time()-t_start:.0f}s")

LLM premise translation failed
Traceback (most recent call last):
  File "/home/phuckhang/MyWorkspace/Exact2026/src/exact/logic/kb.py", line 87, in build_kb_from_premises
    parsed_premises, warnings, predicate_names = translate_premises_only_with_llm(
                                                 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        list(premises), llm_client, settings, temperature=temperature
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/phuckhang/MyWorkspace/Exact2026/src/exact/logic/llm_translator.py", line 271, in translate_premises_only_with_llm
    raw = client.complete_json_sync(messages=messages, temperature=temp, max_tokens=budget)
  File "/home/phuckhang/MyWorkspace/Exact2026/src/exact/llm_client.py", line 196, in complete_json_sync
    return pool.submit(asyncio.run, coro).result()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.14/concurrent/futures/_base.py", line 450, in result
    ret

[   1/1] acc=0.000 ✓0 ✗0 ⚡1 | elapsed=5s ETA=0s

✓ done — 1 predictions in 5s


In [8]:
# ── 7. LƯU PREDICTIONS ───────────────────────────────────────────────────────
output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

output = {
    "model"      : MODEL_NAME,
    "base_url"   : VLLM_BASE_URL,
    "samples"    : TRANSLATION_SAMPLES,
    "limit"      : LIMIT,
    "count"      : len(predictions),
    "format"     : "exact_predictions",
    "predictions": predictions,
}
output_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✓ saved {len(predictions)} predictions → {output_path}")

✓ saved 1 predictions → ../artifacts/predictions/type1/eval_run.json


In [9]:
# ── 8. ĐÁNH GIÁ ──────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class EvalRow:
    id: str | None
    answer: str
    gold_answer: str | None
    question_type: str | None
    status: str
    error: str | None

def _safe_ratio(n: int, d: int) -> float:
    return n / d if d else 0.0

def evaluate_all(preds: list[dict]) -> tuple[list[EvalRow], dict]:
    rows: list[EvalRow] = []
    for p in preds:
        ans      = str(p.get("answer") or "").strip()
        gold     = str(p.get("gold_answer") or "").strip()
        err      = str(p.get("error") or "").strip() or None
        qtype    = str(p.get("question_type") or "unknown")
        if not gold:
            status = "missing_gold"
        elif ans.lower() == gold.lower():
            status = "correct"
        elif err and not ans:
            status = "pipeline_error"
        else:
            status = "wrong"
        rows.append(EvalRow(id=p.get("id"), answer=ans, gold_answer=gold,
                            question_type=qtype, status=status, error=err))

    total         = len(rows)
    missing_gold  = sum(r.status == "missing_gold"  for r in rows)
    scored        = total - missing_gold
    correct       = sum(r.status == "correct"        for r in rows)
    pipe_errors   = sum(r.status == "pipeline_error" for r in rows)
    wrong         = scored - correct

    # Per question type
    grouped: dict[str, list[EvalRow]] = {}
    for r in rows:
        if r.status != "missing_gold":
            grouped.setdefault(r.question_type or "unknown", []).append(r)
    by_qtype = {
        qt: {
            "total"          : len(g),
            "correct"        : sum(r.status == "correct" for r in g),
            "accuracy"       : _safe_ratio(sum(r.status == "correct" for r in g), len(g)),
            "pipeline_errors": sum(r.status == "pipeline_error" for r in g),
        }
        for qt, g in sorted(grouped.items())
    }

    summary = {
        "total"          : total,
        "scored_total"   : scored,
        "correct"        : correct,
        "wrong"          : wrong,
        "missing_gold"   : missing_gold,
        "accuracy"       : _safe_ratio(correct, scored),
        "pipeline_errors": pipe_errors,
        "by_question_type": by_qtype,
    }
    return rows, summary

rows, summary = evaluate_all(predictions)

print("══ SUMMARY ══════════════════════════════")
print(f"  total          : {summary['total']}")
print(f"  scored_total   : {summary['scored_total']}")
print(f"  correct        : {summary['correct']}")
print(f"  wrong          : {summary['wrong']}")
print(f"  pipeline_errors: {summary['pipeline_errors']}")
print(f"  ACCURACY       : {summary['accuracy']:.4f}  ({summary['accuracy']*100:.2f}%)")
print()
print("══ BY QUESTION TYPE ══════════════════════")
for qt, stats in summary["by_question_type"].items():
    print(f"  {qt:<25} acc={stats['accuracy']:.4f}  ({stats['correct']}/{stats['total']})  errors={stats['pipeline_errors']}")

══ SUMMARY ══════════════════════════════
  total          : 1
  scored_total   : 1
  correct        : 0
  wrong          : 1
  pipeline_errors: 1
  ACCURACY       : 0.0000  (0.00%)

══ BY QUESTION TYPE ══════════════════════
  unknown                   acc=0.0000  (0/1)  errors=1


In [10]:
# ── 9. LƯU REPORT VÀ ERRORS CSV ──────────────────────────────────────────────
report_path = Path(REPORT_PATH)
errors_path = Path(ERRORS_PATH)
report_path.parent.mkdir(parents=True, exist_ok=True)
errors_path.parent.mkdir(parents=True, exist_ok=True)

report = {
    "source"   : str(output_path),
    "model"    : MODEL_NAME,
    "count"    : len(rows),
    "summary"  : summary,
    "rows"     : [asdict(r) for r in rows],
}
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✓ report  → {report_path}")

non_correct = [r for r in rows if r.status != "correct"]
if non_correct:
    with errors_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(asdict(non_correct[0]).keys()))
        writer.writeheader()
        writer.writerows(asdict(r) for r in non_correct)
    print(f"✓ errors  → {errors_path}  ({len(non_correct)} rows)")
else:
    print("✓ no errors to write")

✓ report  → ../artifacts/reports/type1_eval_report.json
✓ errors  → ../artifacts/reports/type1_eval_errors.csv  (1 rows)


In [11]:
# ── 10. PHÂN TÍCH LỖI (hiển thị 20 case sai/lỗi đầu tiên) ───────────────────
error_cases = [r for r in rows if r.status != "correct" and r.status != "missing_gold"]
print(f"Non-correct cases: {len(error_cases)}")
print()

for i, row in enumerate(error_cases[:20], 1):
    print(f"[{i:2d}] id={row.id}  type={row.question_type}  status={row.status}")
    print(f"      pred={row.answer!r}  gold={row.gold_answer!r}")
    if row.error:
        print(f"      error={row.error[:120]}")
    print()

Non-correct cases: 1

[ 1] id=logic_0000_00  type=unknown  status=pipeline_error
      pred=''  gold='A'
      error=Type 1 MCQ LLM premise translation failed for request logic_0000_00: All LLM premise translation candidates failed: cand



In [12]:
# ── 11. PHÂN PHỐI CONFIDENCE ─────────────────────────────────────────────────
import statistics

conf_correct = [p["confidence"] for p in predictions if p.get("confidence") and p["_status"] == "correct"]
conf_wrong   = [p["confidence"] for p in predictions if p.get("confidence") and p["_status"] == "wrong"]

def _stats(vals: list[float], label: str) -> None:
    if not vals:
        print(f"  {label}: no data")
        return
    print(f"  {label} (n={len(vals)}):  mean={statistics.mean(vals):.3f}  "
          f"median={statistics.median(vals):.3f}  "
          f"min={min(vals):.3f}  max={max(vals):.3f}")

print("Confidence distribution:")
_stats(conf_correct, "correct")
_stats(conf_wrong,   "wrong  ")

Confidence distribution:
  correct: no data
  wrong  : no data


In [13]:
# ── 12. CHẠY LẠI MỘT SAMPLE CỤ THỂ (debug) ──────────────────────────────────
# Thay TARGET_ID bằng ID muốn debug, ví dụ: "logic_0000_00"
TARGET_ID = "logic_0000_00"

ex = next((e for e in examples if e.request.id == TARGET_ID), None)
if ex is None:
    print(f"ID '{TARGET_ID}' không tìm thấy trong dataset")
else:
    route = router.route(ex.request)
    resp  = run_type1_pipeline(
        ex.request,
        translator_client=translator_client,
        settings=settings,
        question_type=route.question_type,
    )
    print(f"id            : {resp.id}")
    print(f"question_type : {route.question_type.value}")
    print(f"answer        : {resp.answer}")
    print(f"gold_answer   : {ex.gold_answer}")
    print(f"confidence    : {resp.confidence}")
    print(f"explanation   : {resp.explanation}")
    print(f"fol           : {resp.fol}")
    print(f"cot           :")
    for step in (resp.cot or []):
        print(f"  - {step}")
    if resp.error:
        print(f"error         : {resp.error}")

LLM premise translation failed
Traceback (most recent call last):
  File "/home/phuckhang/MyWorkspace/Exact2026/src/exact/logic/kb.py", line 87, in build_kb_from_premises
    parsed_premises, warnings, predicate_names = translate_premises_only_with_llm(
                                                 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        list(premises), llm_client, settings, temperature=temperature
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/phuckhang/MyWorkspace/Exact2026/src/exact/logic/llm_translator.py", line 271, in translate_premises_only_with_llm
    raw = client.complete_json_sync(messages=messages, temperature=temp, max_tokens=budget)
  File "/home/phuckhang/MyWorkspace/Exact2026/src/exact/llm_client.py", line 196, in complete_json_sync
    return pool.submit(asyncio.run, coro).result()
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.14/concurrent/futures/_base.py", line 450, in result
    ret

RuntimeError: Type 1 MCQ LLM premise translation failed for request logic_0000_00: All LLM premise translation candidates failed: candidate 1/3 failed: LLM premise translation failed: Client error '400 Bad Request' for url 'https://api.iamphuckhang.dev/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400; candidate 2/3 failed: LLM premise translation failed: Client error '400 Bad Request' for url 'https://api.iamphuckhang.dev/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400; candidate 3/3 failed: LLM premise translation failed: Client error '400 Bad Request' for url 'https://api.iamphuckhang.dev/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400